# Stage 2 Notebook 63 - Exp2HHH DN-DETR + ultra-stable training (lr=1e-4, wd=1e-3, clip=2)

**Aggressive stabilization of the DN-DETR head.** NB59 confirmed the architecture can hit val_lane_f1=0.62 at epoch 3 but oscillates wildly between 0.22 and 0.62. The DAB anchor parameters drift between stable configurations due to high effective LR.

Exp2HHH: brute-force stabilization via training-side hyperparameters:
- `lr0: 0.0002 -> 0.0001` (halve main LR)
- `backbone_lr_mult: 0.02 -> 0.005` (4x slower backbone, 200x slower than baseline)
- `weight_decay: 0.0005 -> 0.001` (2x stronger weight regularization)
- `grad_clip_norm: 5.0 -> 2.0` (tighter gradient clipping)
- `head_warmup until_epoch: 10 -> 12` (longer head-only training)
- `lambda_det: 1.0 -> 1.5` (slight det boost to avoid the late-epoch det collapse NB59 showed)
- `use_uncertainty: true -> false`

Hypothesis: DAB anchors move slowly enough to stay near optimum. Should give STABLE val_lane_f1 ≥ 0.50 from epoch 10 onwards. Wall-clock similar to NB59 (~35 min).

### Run mode
1. Smoke.
2. 20 epochs limit=3000.

In [1]:
import os, sys, subprocess, textwrap
from google.colab import drive
os.environ['PYTHONUNBUFFERED'] = '1'
drive.mount('/content/drive')

REPO_ROOT = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
if not os.path.isdir(REPO_ROOT):
    raise FileNotFoundError(f'Missing project root: {REPO_ROOT}')
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml', 'scipy', 'opencv-python-headless', 'tqdm', 'matplotlib'])
print('repo:', REPO_ROOT)

from stage2.scripts.notebook_utils import run_streaming
LOG_DIR = '/content/drive/MyDrive/EcoCAR/training_runs/notebook_logs'
os.makedirs(LOG_DIR, exist_ok=True)

Mounted at /content/drive
repo: /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane


In [2]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp58_rmt_gca_query64_dn_vfl_stable_joint.yaml'
LOG_FILE = os.path.join(LOG_DIR, f'{Path(CONFIG).stem}_smoke.log')
run_streaming([sys.executable, '-u', 'stage2/scripts/smoke_test_joint_models.py', CONFIG], log_path=LOG_FILE)

[run_streaming] command: /usr/bin/python3 -u stage2/scripts/smoke_test_joint_models.py stage2/configs/exp58_rmt_gca_query64_dn_vfl_stable_joint.yaml
[run_streaming] log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp58_rmt_gca_query64_dn_vfl_stable_joint_smoke.log
OK exp58_rmt_gca_query64_dn_vfl_stable_joint.yaml
  lane_shape=(1, 64, 72, 2) det_shape=(1, 4, 4)
  lane_loss=4.4840 det_loss=3.2828 grad_cos=-0.2117 lambda_lane=0.0500
  gate_stats={'gate/det_mean': 0.4978388547897339, 'gate/lane_mean': 0.5004215836524963, 'gate/det_sat_low': 0.0, 'gate/det_sat_high': 0.0, 'gate/lane_sat_low': 0.0, 'gate/lane_sat_high': 0.0}
[run_streaming] return_code=0


0

In [3]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp58_rmt_gca_query64_dn_vfl_stable_joint.yaml'
CURVE_TAR = '/content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar'
CURVE_ROOT = '/content/bdd100k_clrkd_curve'

DEBUG_MODE = False

if DEBUG_MODE:
    RUN_TAG = 'debug'
    EPOCHS = 2
    BATCH_SIZE = 4
    LIMIT_TRAIN = 512
    LIMIT_VAL = 256
    PRINT_EVERY = 5
else:
    RUN_TAG = 'short20'
    EPOCHS = 20
    BATCH_SIZE = 8
    LIMIT_TRAIN = 3000
    LIMIT_VAL = 1000
    PRINT_EVERY = 50

run_stem = Path(CONFIG).stem + '_' + RUN_TAG
WORK_DIR = f'/content/{run_stem}'
OUTPUT_TAR = f'/content/drive/MyDrive/EcoCAR/training_runs/{run_stem}.tar'
LOG_FILE = os.path.join(LOG_DIR, f'{run_stem}_train.log')

cmd = [
    sys.executable, '-u', 'stage2/scripts/train_joint_model_experiment.py',
    '--config', CONFIG,
    '--curve-tar', CURVE_TAR,
    '--curve-root', CURVE_ROOT,
    '--work-dir', WORK_DIR,
    '--output-tar', OUTPUT_TAR,
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--limit-val', str(LIMIT_VAL),
    '--force-extract',
    '--print-every', str(PRINT_EVERY),
]
if LIMIT_TRAIN is not None:
    cmd.extend(['--limit-train', str(LIMIT_TRAIN)])

print('DEBUG_MODE:', DEBUG_MODE, flush=True)
print('LIMIT_TRAIN:', LIMIT_TRAIN, flush=True)
print('About to run:', ' '.join(cmd), flush=True)
print('Output tar:', OUTPUT_TAR, flush=True)
print('Visible log file:', LOG_FILE, flush=True)
run_streaming(cmd, log_path=LOG_FILE)

DEBUG_MODE: False
LIMIT_TRAIN: 3000
About to run: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp58_rmt_gca_query64_dn_vfl_stable_joint.yaml --curve-tar /content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar --curve-root /content/bdd100k_clrkd_curve --work-dir /content/exp58_rmt_gca_query64_dn_vfl_stable_joint_short20 --output-tar /content/drive/MyDrive/EcoCAR/training_runs/exp58_rmt_gca_query64_dn_vfl_stable_joint_short20.tar --epochs 20 --batch-size 8 --limit-val 1000 --force-extract --print-every 50 --limit-train 3000
Output tar: /content/drive/MyDrive/EcoCAR/training_runs/exp58_rmt_gca_query64_dn_vfl_stable_joint_short20.tar
Visible log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp58_rmt_gca_query64_dn_vfl_stable_joint_short20_train.log
[run_streaming] command: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp58_rmt_gca_query64_dn_vfl_stable_joint.yaml --curve-t

0

## What to watch in Exp2HHH

Reference NB59 (DN-DETR + 10ep warmup): val_lane_f1 oscillates 0.22-0.62, peak ep3=0.623.

Pass criteria at epoch 20:
- **val_lane_f1 >= 0.40 SUSTAINED from epoch 10 onwards** -- no oscillation = training-side stabilization worked.
- val_lane_best_f1 >= 0.50.
- gap >= 0.15 at epoch 20.
- matched_iou >= 0.20 (geometry still trained though slower).
- val_det <= 2.5 (det not crashed).

If sustained val_lane_f1 >= 0.40 + matched_iou >= 0.20: we have a DEPLOYABLE query head model with the cls breakthrough preserved. Combine with anchor's curves at inference (anchor coord_pred + query cls scoring) for the final fusion model.

If still oscillates: need code-level intervention (EMA model averaging or static anchor positions).